# 01 — Explore EvaLatin 2024 Parsing Data

Load test/gold CoNLL-U, see what's in the files, look at distributions and a real tree.

Plain-English background: [`../docs/00_task_explained.md`](../docs/00_task_explained.md).

In [ ]:
import conllu
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from latinbench.data import test_path, gold_path

## Test vs gold — same first sentence

In [ ]:
first_test = test_path('poetry').read_text().split('\n\n')[0]
first_gold = gold_path('poetry').read_text().split('\n\n')[0]
print('--- TEST (HEAD + DEPREL are _) ---')
print(first_test)
print()
print('--- GOLD ---')
print(first_gold)

## Load with `conllu`

In [ ]:
def load(p):
    return conllu.parse(p.read_text())

poetry = load(gold_path('poetry'))
prose  = load(gold_path('prose'))
print(f'poetry: {len(poetry)} sentences')
print(f'prose:  {len(prose)} sentences')

## Basic stats

In [ ]:
def stats(sents, name):
    n_tok = sum(len(s) for s in sents)
    return {'split': name, 'sentences': len(sents), 'tokens': n_tok, 'avg_sent_len': round(n_tok / len(sents), 1)}

pd.DataFrame([stats(poetry, 'poetry'), stats(prose, 'prose')])

## UPOS distribution

In [ ]:
def upos_dist(sents):
    return Counter(t['upos'] for s in sents for t in s)

df = pd.DataFrame({'poetry': upos_dist(poetry), 'prose': upos_dist(prose)}).fillna(0).astype(int)
df['total'] = df.sum(axis=1)
df = df.sort_values('total', ascending=False)
df

In [ ]:
ax = df[['poetry', 'prose']].plot(kind='bar', figsize=(10, 4))
ax.set_title('UPOS — poetry vs prose')
ax.set_ylabel('count')
plt.tight_layout(); plt.show()

## DEPREL distribution (top 20)

In [ ]:
def deprel_dist(sents):
    return Counter(t['deprel'] for s in sents for t in s)

dr = pd.DataFrame({'poetry': deprel_dist(poetry), 'prose': deprel_dist(prose)}).fillna(0).astype(int)
dr['total'] = dr.sum(axis=1)
dr = dr.sort_values('total', ascending=False).head(20)
dr

## A real dependency tree

In [ ]:
short = next(s for s in poetry if 6 <= len(s) <= 9)
print('# text:', short.metadata.get('text', ''))
print()
short.to_tree().print_tree()

## Sentence length distribution

In [ ]:
lens_poetry = [len(s) for s in poetry]
lens_prose  = [len(s) for s in prose]
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist([lens_poetry, lens_prose], bins=30, label=['poetry', 'prose'])
ax.set_xlabel('sentence length (tokens)')
ax.set_ylabel('count')
ax.legend()
plt.tight_layout(); plt.show()
print(f'poetry: median {np.median(lens_poetry):.0f}, max {max(lens_poetry)}')
print(f'prose:  median {np.median(lens_prose):.0f}, max {max(lens_prose)}')

## Next

Open [`02_compare_models.ipynb`](02_compare_models.ipynb) to run the baseline + LatinPipe + your own model.